In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
from default_risk.scripts.auxiliar_eda_function import check_invariant
import default_risk.config as cfg
import dtale
import logging
import dtale.global_state as dtale_global
import gc


dtale_global.cleanup()
gc.collect()


log = logging.getLogger('werkzeug')


bureau_balance_df= pd.read_csv(cfg.BUREAU_BALANCE)

column_order_reference= "MONTHS_BALANCE"

data_frame_size=len(bureau_balance_df)

amount_of_loans= bureau_balance_df["SK_ID_BUREAU"].nunique()

bureau_balance_df.sort_values(["SK_ID_BUREAU",column_order_reference],inplace=True)

#aux_function

def get_full_sorted_serie_rows(rows : pd.DataFrame) -> pd.DataFrame:
   return recreate_and_sort_series_given_rows(rows,bureau_balance_df, "SK_ID_BUREAU",column_order_reference)

def get_full_sorted_serie_ids(ids : list) -> pd.DataFrame:
   return recreate_and_sort_the_serie_given_ids(ids,bureau_balance_df, "SK_ID_BUREAU" ,column_order_reference)


with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

Invariants found at the moment: (# number of cell with the proofs and relevant code)

1- The MONTHS_BALANCE counter contains no gaps. Formally speaking, for an SK_ID_BUREAU and its respective MONTHS_BALANCE.max_value and MONTHS_BALANCE.min_value, there exists one row for every value between the minimum and maximum MONTHS_BALANCE values for that SK_ID_BUREAU (100%) #8

Soft constraints:

1- Once the temporal sequence is marked as closed (STATUS == "C") the remaining rows of the sequence are also marked as closed until the last record ("MONTHS_BALANCE" == -1) (99.9%)
    Anomalies: Potential instances of data corruption. #6

Decisions summary:  

1- Closed loan detection #5: in accordance with the first soft contraint, we define this rule to know when the loan is closed and consider the rest as "padding": 
    I- two consecutive rows contain STATUS == "C", or
    II- a single row with status closed but being the last record of the sequence. 
    This rule is created with the intention to be robust against glitches we detect associated with labeling errors that are reverted in the next row. We can capture the next
    values of days past due. 
    Otherwise, if the loan is in fact closed and the later rows contain delinquency information after that moment, we can ignore it assuming data corruption. 

2- Padding #4, #6: Most loans (78.2%) have padding after closure until the most recent months. 
    This means once a loan is marked as closed, additional records keep appearing until the most recent month (around MONTH_BALANCE = -1) that don't provide information of the payment behavior.
    therefore, we will create a flag/counter to track the cases that don't follow this pattern (incomplete/missing padding)

3- Non-closed sequences #7: 
We define as non-closed sequences the ones that don't have any row with closed status (STATUS == "C")
and with two particular subcases:
    1- potential ongoing loan (when the most recent record has MONTH_BALANCE > -3). 
    2- incomplete sequence (MONTH_BALANCE < -3) 
    note: The ongoing case seems to include more cases that should because the dataset has a lot of status "X" (unknown) and based 
        on the semantic some loans may actually be closed and we are not able to determine it because the status "C" was replaced with an "X". However we catch this information
        with the intention to check how the parent table (Bureau) can help to clarify the pattern. 


In [ ]:
#create the files por data data dictionary  #1
create_files_nulls_per_colmun(bureau_balance_df,"bureu_balance")

In [ ]:
#run the screening script on bureau_balance #2
eda_per_table_printing_results(bureau_balance_df, schema, "bureau_balance",False)

In [ ]:
#3
ids_bureau=bureau_balance_df["SK_ID_BUREAU"].unique()
first_ids=ids_bureau[:100]
first_series= get_full_sorted_serie_ids(first_ids)
dtale.show(first_series)

In [ ]:
print(bureau_balance_df["SK_ID_BUREAU"].nunique())

In [ ]:
#4
sns.displot(data=bureau_balance_df,bins=1,discrete=True,kind="hist", x="MONTHS_BALANCE",col="STATUS")

In [9]:
#5
bureau_balance_df["NEXT_STATUS"] = bureau_balance_df.groupby("SK_ID_BUREAU")["STATUS"].shift(-1)
viable_status_mask = ((bureau_balance_df["NEXT_STATUS"] != "C") & (bureau_balance_df["NEXT_STATUS"].notna())) 
reopen_rows=bureau_balance_df[(bureau_balance_df["STATUS"] == "C") & (viable_status_mask)]
dtale.show(get_full_sorted_serie_rows(reopen_rows))

In [ ]:
#6

most_recent_row= bureau_balance_df.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].transform("max")
rows_without_full_pading= bureau_balance_df[most_recent_row < -3]


check_invariant((bureau_balance_df["STATUS"] == "C") & (viable_status_mask),"status C is reverted",data_frame_size)

check_invariant((most_recent_row < -3),"the padding don't get until -1",data_frame_size)

print((rows_without_full_pading["SK_ID_BUREAU"].nunique() * 100) / (amount_of_loans))

dtale.show(rows_without_full_pading)


52 of cases where status C is reverted
that represent a 0.00019047671376386566% of cases with violation of this invariant 

4543882 of cases where the padding don't get until -1
that represent a 16.64430213636118% of cases with violation of this invariant 

21.87706066222573


In [ ]:

#7
have_at_least_one_status_closed= bureau_balance_df["STATUS"].eq("C").groupby(bureau_balance_df["SK_ID_BUREAU"]).transform("any")
have_recent_balance= bureau_balance_df["MONTHS_BALANCE"].gt(-4).groupby(bureau_balance_df["SK_ID_BUREAU"]).transform("any")

incompleted_series_df= bureau_balance_df[(~have_at_least_one_status_closed) & (~have_recent_balance)]
incomplete_amount_of_series= incompleted_series_df["SK_ID_BUREAU"].nunique()
porcentaje_of_incompleted_series= (incomplete_amount_of_series * 100) / amount_of_loans


on_going_series_df= bureau_balance_df[(~have_at_least_one_status_closed) & (have_recent_balance)]
amount_of_ongoing_series= on_going_series_df["SK_ID_BUREAU"].nunique()
porcentaje_of_ongoing_series= (amount_of_ongoing_series * 100) / amount_of_loans


print(str(porcentaje_of_incompleted_series) + "%" + " of the series from the dataset are incompleted series")
print(str(porcentaje_of_ongoing_series) + "%" + " are potential ongoing loans")

dtale.show(on_going_series_df)



10.758201359195983% of the series from the dataset are incompleted series
34.23730265049333% are ongoing loans


In [ ]:
#8
diff = (bureau_balance_df["MONTHS_BALANCE"]- bureau_balance_df.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].shift(1))

gaps = bureau_balance_df[(diff.notna()) & (diff !=1)]

len(gaps)

0

In [ ]:
pd.set_option("display.max_columns", None)        # mostrar todas las columnas
pd.set_option("display.max_rows", None)            # cantidad máxima de filas visibles
pd.set_option("display.width", None)              # evitar wrap artificial
pd.set_option("display.max_colwidth", None)       # no truncar strings
pd.set_option("display.expand_frame_repr", False) # evitar salto horizontal raro
pd.set_option("display.float_format", "{:.4f}".format)  # floats legibles

In [ ]:
print(reopen_rows["SK_ID_BUREAU"].nunique())
